## Rag From Scratch: Query Transformations/Transilation

![alt text](<../../assets/Full Rag Pipeline.png>)

### Environment

In [8]:
import warnings
import os 
from dotenv import load_dotenv

# 0. Disable Warnings
warnings.filterwarnings("ignore")

# 1. Add parentheses to actually run the function
load_dotenv()

try: 
    # 2. Use .get() with a default empty string "" to avoid NoneType errors
    os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGSMITH_TRACING_V2")
    os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
    os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGSMITH_PROJECT")
    os.environ["LANGCHAIN_ENDPOINT"] = os.getenv("LANGSMITH_ENDPOINT")
    os.environ["MISTRAL_API_KEY"] = os.getenv("MISTRAL_API_KEY")
    os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
    os.environ["USER_AGENT"] = "MyLangChainApp/1.0" # For WebBaseLoader
    print("Environment variables set successfully")
except Exception as e: 
    print(f"Error: {e}")

Environment variables set successfully


### Part 5: Multi Query

Flow:

![alt text](../../assets/MultiQuery.png)

#### Indexing

In [9]:
import bs4
# from langchain.text_splitter import RecursiveCharacterTextSplitter # Old import
from langchain_text_splitters import RecursiveCharacterTextSplitter # New import
from langchain_community.document_loaders import WebBaseLoader # New import
from langchain_community.vectorstores import Chroma # New import
from langchain_core.output_parsers import StrOutputParser # New import
from langchain_core.runnables import RunnablePassthrough # New import
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings # New import
from pathlib import Path
from langsmith import Client
client = Client()

# .parent.parent moves up twice to reach the root (rag_tutorial)
ROOT_DIR = Path.cwd().parents[1]

# Define the exact folder name at the root level
DB_DIR = ROOT_DIR / "db_blog"

In [10]:
# #### INDEXING ####

# Load blog
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

# Split 
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50,
)

# Make Splits
splitted_docs = splitter.split_documents(blog_docs)


# Index
# There is no need to re-store these in the vector store, so we just gonna use the one we have.
embeddings = MistralAIEmbeddings(model = "mistral-embed")
vectorstore = Chroma.from_documents(
    documents = splitted_docs,
    embedding = embeddings,
    collection_name = "blog_posts", 
    persist_directory = str(DB_DIR)
)
retriever = vectorstore.as_retriever(search_kwargs = {"k":3}) 
# the number of the search_kwargs which is the number of most relevant document to retrieve 
# this number controls the documents in the langSmith trace view

#### Prompt

In [11]:
from langchain_core.prompts import ChatPromptTemplate

# Multi Query: Different Perspectives
template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. Do not include any preamble.
Original question: {question}"""

# Prompt Template
prompt_perspectives = ChatPromptTemplate.from_template(template)

generate_query_chain = (
    prompt_perspectives 
    | ChatMistralAI(model = "mistral-small-latest", temperature=0)
    | StrOutputParser()
    | (lambda x: [q for q in x.split("\n") if q.strip()])
)

# x going to split the query into multiple queries as a list
# the q it self doing a clean for each line in the list item by item.x``

In [12]:
from langchain_core.load import loads, dumps

def get_unique_union(documents: list[list]): 
    """ Unique union of retrieved docs """
    # Flatten list of lists, and convert each Document to string
    flatten_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Get unique documents
    unique_docs = list(set(flatten_docs))
    # Return list of Document objects
    return [loads(docs) for docs in unique_docs]


# Retrieve
question = "What is task decomposition for LLM agents?"
retrieval_chain = (
    generate_query_chain | retriever.map() | get_unique_union)

docs = retrieval_chain.invoke({"question": question})

len(docs)



3

In [13]:
# Limit to the first 3 documents to keep your terminal output clean
for i, doc in enumerate(docs):
    print(f"\n{'='*40}")
    print(f"📄 DOCUMENT {i+1}")
    print(f"{'='*40}")
    
    # Safely extract the source URL from the metadata dictionary
    # We use .get() so it doesn't crash if the 'source' key is missing
    source = doc.metadata.get('source', 'Unknown Source')
    print(f"🔗 Source: {source}\n")
    
    # Print the first 400 characters of the page content to verify the text chunks
    print("📝 Content Preview:")
    print(f"{doc.page_content[:400]}...")


📄 DOCUMENT 1
🔗 Source: https://lilianweng.github.io/posts/2023-06-23-agent/

📝 Content Preview:
LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays an...

📄 DOCUMENT 2
🔗 Source: https://lilianweng.github.io/posts/2023-06-23-agent/

📝 Content Preview:
(2) Model selection: LLM distributes the tasks to expert models, where the request is framed as a multiple-choice question. LLM is presented with a list of models to choose from. Due to the limited context length, task type based filtration is needed.
Instruction:

Given the user request and the call command, the AI assistant helps the user to select a suitable model from a list of models to proce..

In [14]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
from langchain_mistralai import ChatMistralAI

# RAG
template = """Answer the following question based on this context: 

Context: {context}


Question: {question}
""" 

prompt = ChatPromptTemplate.from_template(template)

llm = ChatMistralAI(model = "mistral-small-latest", temperature = 0)

final_rag_chain = (
    {"context": retrieval_chain, 
    "question": itemgetter("question")} 
    | prompt 
    | llm
    | StrOutputParser()
)

docs = final_rag_chain.invoke({"question": "What is the main topic of the blog post?"})

### Part 6: Rag Fusion

Flow:

![alt text](<../../assets/Rag Fusion.png>)

#### Prompt

In [23]:
from langchain_core.prompts import ChatPromptTemplate

# RAG-Fusion: Related
template = """You are a helpful assistant that generates multiple search queries based on a single input query without any preamble. \n
Generate multiple search queries related to: {question} \n
Output (4 queries):"""

prompt_rag_fusion = ChatPromptTemplate.from_template(template)

In [25]:
from langchain_core.output_parsers import StrOutputParser
from langchain_mistralai import ChatMistralAI

llm = ChatMistralAI(model = "mistral-small-latest")

generated_queries = (
    prompt_rag_fusion
    | llm
    | StrOutputParser()
    | (lambda x: [q for q in x.split("\n") if q.strip()])
    
)

In [ ]:
def reciprocal_rank_fusion(results: list[list], k=60):
    """ Reciprocal_rank_fusion that takes multiple lists of ranked documents 
        and an optional parameter k used in the RRF formula """
    
    # Initialize a dictionary to hold fused scores for each unique document
    fused_scores = {}
    # Iterate through each list of ranked documents
    for docs in results:
        # Iterate through each document in the list, with its rank (position in the list)
        for rank, doc in enumerate(docs):
            # Convert the document to a string format to use as a key (assumes documents can be serialized to JSON)
            doc_str = dumps(doc)
            # If the document is not yet in the fused_scores dictionary, add it with an initial score of 0
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            # Retrieve the current score of the document, if any
            previous_score = fused_scores[doc_str]
            # Update the score of the document using the RRF formula: 1 / (rank + k)
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort the documents based on their fused scores in descending order to get the final reranked results
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    # Return the reranked results as a list of tuples, each containing the document and its fused score
    return reranked_results

retrieval_chain_rag_fusion =(
    generated_queries
    | retriever.map() 
    | reciprocal_rank_fusion
)

docs = retrieval_chain_rag_fusion.invoke({"question": question})
len(docs)

2

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# RAG
template = """Answer the following question based on this context: 
{context}


Question: {question}
"""

# Prompt
prompt = ChatPromptTemplate.from_template(template)


# LLM
llm = ChatMistralAI(model="mistral-small-latest")

# RUN

question = question = "What is task decomposition for LLM agents?"

final_fusion_rag_chain = (
    {"context": retrieval_chain_rag_fusion,
    "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

final_fusion_rag_chain.invoke({"question": question})

'Task decomposition for LLM (Large Language Model) agents refers to the process of breaking down a complex task into smaller, more manageable sub-tasks or steps. This technique helps the agent systematically approach and solve complicated tasks by simplifying them into a sequence of easier-to-handle components.\n\n### Key Methods of Task Decomposition:\n1. **Chain of Thought (CoT)**:\n   - The model is prompted to "think step by step," decomposing a complex task into smaller, sequential steps.\n   - This enhances the model\'s reasoning by leveraging more computational effort during inference.\n\n2. **Tree of Thoughts (ToT)**:\n   - Extends CoT by exploring multiple reasoning paths at each step, creating a tree-like structure of possible solutions.\n   - The search can be done using breadth-first search (BFS) or depth-first search (DFS), with evaluations via a classifier or majority voting.\n\n3. **Prompting Techniques**:\n   - Simple prompts like *"Steps for XYZ. 1."* or *"What are the

### Part 7: Decomposition

In [33]:
from langchain_community.vectorstores import Chroma # New import
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings
from langchain_core.output_parsers import StrOutputParser
from pathlib import Path
# Setup the Embeddings
embeddings = MistralAIEmbeddings(model="mistral-embed")

# Setup the Retriever with the DB path

# .parent.parent moves up twice to reach the root (rag_tutorial)
ROOT_DIR = Path.cwd().parents[1]

# Define the exact folder name at the root level
DB_DIR = ROOT_DIR / "db_blog"

try: 
    # Open a DB (connect)
    vectorstore = Chroma(
        embedding_function = embeddings,
        collection_name = "blog_posts", 
        persist_directory = str(DB_DIR)
    )
    
    # Check the number of items in the collection

    count = vectorstore._collection.count()
    print(f"Total items in 'blog_posts': {count}")
    if count > 0:
        print("Database opened successfully with existing data.")
except ConnectionError as e: 
    print("Database is empty. Check your path or if data was previously saved. The DB will be Initialized.")
    
retriever = vectorstore.as_retriever()

Total items in 'blog_posts': 350
Database opened successfully with existing data.


In [43]:
from langchain_core.prompts import ChatPromptTemplate

template = """You are a helpful assistant that generates multiple sub-questions related to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that can be answers in isolation without preambles. \n
Generate multiple search queries related to: {question} \n
Output (3 queries):"""

prompt_decomposition = ChatPromptTemplate.from_template(template)

# LLM
llm = ChatMistralAI(model = "mistral-small-latest",
                    temperature=0)

# Chain
generate_query_decomposition = (
    prompt_decomposition
    | llm
    | StrOutputParser()
    | (lambda x: [q for q in x.split("\n") if q.strip()])
)
# Run
question = "What are the main components of an LLM-powered autonomous agent system?"
queries = generate_query_decomposition.invoke({"question": question})

print("\n".join(queries))

1. **"What are the core components of an LLM-powered autonomous agent system?"**
2. **"What are the key modules required for an autonomous agent using large language models?"**
3. **"How do LLM-based autonomous agents integrate different system components?"**


####  Answer recursively

![alt text](../../assets/Decomposition_answer_recursivly.png)

In [51]:
# Prompt
template = """Here is the question you need to answer:

\n --- \n {question} \n --- \n

Here is any available background question + answer pairs:

\n --- \n {q_a_pairs} \n --- \n

Here is additional context relevant to the question: 

\n --- \n {context} \n --- \n

Use the above context and any background question + answer pairs to answer the question: \n {question}
"""

decomposition_prompt = ChatPromptTemplate.from_template(template)

In [52]:
from operator import itemgetter

def format_qa_pair(question, answer): 
    """ Format Q and A pair"""
    
    formatted_string = ""
    formatted_string += f"Question: {question}\nAnswer: {answer}\n\n"
    return formatted_string.strip()
# LLM 
llm = ChatMistralAI(model = "mistral-small-latest")

q_a_pairs = ""

for q in queries:
    rag_chain = (
        {
        "context": itemgetter("question") | retriever, 
        "question": itemgetter("question"),
        "q_a_pairs": itemgetter("q_a_pairs")
        } 
    | decomposition_prompt
    | llm
    | StrOutputParser()
    )
    answer = rag_chain.invoke({"question": q, "q_a_pairs": q_a_pairs})
    q_a_pair = format_qa_pair(q, answer)
    q_a_pairs = q_a_pairs + "\n---\n" + q_a_pair
    print(q, ":\n", answer, "\n ")  



1. **"What are the core components of an LLM-powered autonomous agent system?"** :
 The core components of an LLM-powered autonomous agent system, as described in the provided context, include:

1. **LLM (Large Language Model) as the Core Controller**: The LLM serves as the "brain" of the agent, enabling it to process information, generate responses, and make decisions.

2. **Planning**:
   - **Subgoal and Decomposition**: The agent breaks down complex tasks into smaller, manageable subgoals to handle them efficiently.
   - **Reflection and Refinement**: The agent can self-critique and refine its actions based on past performance, improving future steps and outcomes.

3. **Memory**: The agent retains information from past interactions or tasks, allowing it to learn and adapt over time.

These components work together to enable the agent to function as a general problem solver, extending beyond simple text generation to more complex, autonomous tasks. Examples of such systems include Au

#### Answer individually

![alt text](../../assets/Decomposition_Answer_individually.png)

In [47]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langsmith import Client
client = Client()

# Pull the prompt
prompt_rag = client.pull_prompt("rlm/rag-prompt")

def retrieve_and_rag(question, prompt_rag, sub_question_generator_chain):
    """RAG on each sub-question"""
    
    # Use our Decomposition 
    sub_questions = sub_question_generator_chain.invoke({"question":question})
    
    # initialize a list hold rag results 
    rag_results = []
    
    for sub_question in sub_questions: 
        # Retrieve documents for each sub-question
        retrieved_docs = retriever.invoke(sub_question)
        
        # Use retrieved documents and sub-question in RAG chain
        answer = (
            prompt_rag 
            | llm 
            | StrOutputParser()
        ).invoke({
                "question": sub_question,
                "context": retrieved_docs})
        # Append the result into the rag_result list
        rag_results.append(answer)
    return rag_results, sub_questions

# Wrap the retrieval and RAG process in a RunnableLambda for integration into a chain
answers, questions = retrieve_and_rag(question, prompt_rag, generate_query_decomposition)

In [48]:
def format_qa_pairs(questions, answers):
    """Format Q and A pairs"""
    
    formatted_string = ""
    for i, (question, answer) in enumerate(zip(questions, answers), start = 1):
        formatted_string += f"Question {i}: {question}\nAnswer {i}: {answer}\n\n"
        
    return formatted_string.strip()

context = format_qa_pairs(questions, answers)

# Prompt
template = """Here is a set of Q+A pairs:

{context}

Use these to synthesize an answer to the question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"context":context,"question":question})

'The main components of an **LLM-powered autonomous agent system** include:\n\n1. **Large Language Model (LLM)** – Acts as the core "brain" of the system, processing inputs, generating responses, and orchestrating decision-making.\n2. **Planning Module** – Enables task breakdown through **subgoal decomposition** and **reflection/refinement**, allowing the agent to handle complex tasks systematically.\n3. **Memory** – Stores past actions, experiences, and learned insights, enabling the agent to improve over time by learning from mistakes and adapting strategies.\n\nThese components work together to allow the agent to autonomously decompose tasks, execute actions, reflect on outcomes, and refine its approach for better performance. The LLM serves as the central controller, while planning and memory enhance its ability to solve problems efficiently and iteratively.'